# sceneid benchmark (Tier 1) on Colab

This notebook runs the whole benchmark: it downloads the 18 open films, cuts and distorts about
6,200 test clips, embeds everything with CLIP on Colab's GPU, and scores V1 against V2.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

**Where things go**
- Films and temporary clips: `/content/bench` (Colab's own disk, about 8 GB, lost when the session ends).
- Everything worth keeping: `MyDrive/sceneid-bench` on your Google Drive (under 1 GB): the answer key,
  the library, the query embeddings and the results.

**If Colab disconnects:** run the cells from the top again. Every step skips work that's already done and
saved on Drive, so you lose a few minutes at most. Keep this tab open while it runs.

**Time:** about 10 minutes of setup and downloads, then roughly 1 to 2 hours for step 5.

## 1. Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

EMBEDDER = "clip-vit-b32"
BRANCH = "main"
WORK = "/content/bench"  # films, temporary clips
DRIVE = "/content/drive/MyDrive/sceneid-bench"  # everything kept
os.makedirs(DRIVE, exist_ok=True)

os.environ["SCENEID_BATCH_SIZE"] = "256"  # frames per GPU batch
os.environ["SCENEID_LOG_LEVEL"] = "INFO"

COMMON = (
    f"--workspace {WORK} --queries {DRIVE}/queries.jsonl "
    f"--library-dir {DRIVE}/libraries/{EMBEDDER} --embeddings-dir {DRIVE}/embeddings/{EMBEDDER}"
)
BENCH = f"sceneid --embedder {EMBEDDER} bench {COMMON}"
print(BENCH)

## 2. Install sceneid

In [ ]:
if not os.path.isdir("/content/sceneid"):
    !git clone --depth 1 -b {BRANCH} https://github.com/saarvesh0606/clip-to-video-scene-id.git /content/sceneid
else:
    !git -C /content/sceneid pull --ff-only
%cd /content/sceneid
!pip install -q -e ".[clip,bench]"
!ffmpeg -version | head -n 1
!git -C /content/sceneid log --oneline -1

## 3. Download the films (about 7 GB)

Each file is checked against its published size, and the Internet Archive files against their MD5
checksums. The licences are listed in `benchmarks/datasets/tier1.json`.

In [ ]:
!{BENCH} download

## 4. Build the answer key and the library

The answer key is built once and kept on Drive, so every rerun (and every future model) is scored
on exactly the same clips. The library indexes the 13 library films at 2 frames per second.

In [ ]:
if not os.path.exists(f"{DRIVE}/queries.jsonl"):
    !{BENCH} queries --seed 0
else:
    print("answer key already on Drive: keeping it")
!{BENCH} index

## 5. Embed every query clip

Worker threads cut and distort each clip with ffmpeg; the GPU embeds frames in large batches; each
clip is deleted after it's embedded. Progress is saved to Drive every 200 clips.

Run the trial cell first to see the speed and time remaining, then the full cell.
**If the session drops, reconnect, run steps 1–3 again, then this cell. It picks up where it stopped.**

In [ ]:
!{BENCH} embed --limit 40 --workers 2

In [ ]:
!{BENCH} embed --workers 2
!{BENCH} status

## 6. Score it

This runs on the CPU and takes a few minutes. It compares three setups: V1 with its original
thresholds (0.83 / 0.90), V1 with thresholds tuned on the val split, and V2 tuned the same way. All
numbers in the report come from the test split.

In [ ]:
RESULTS = f"{DRIVE}/results/tier1-{EMBEDDER}"
!{BENCH} evaluate --out {RESULTS}
from IPython.display import Image, Markdown, display

display(Markdown(open(f"{RESULTS}/report.md").read()))
display(Image(f"{RESULTS}/roc.png"))

## 7. Bring the results back

The results are on your Drive in `sceneid-bench/results/tier1-clip-vit-b32/`. Download that folder
and put it in the repo at `benchmarks/results/tier1-clip-vit-b32/`, or run the cell below to get
a zip. The library and embeddings (also on Drive) let you re-score later without a GPU.

In [ ]:
import shutil

from google.colab import files

zip_path = shutil.make_archive("/content/tier1-results", "zip", RESULTS)
files.download(zip_path)